In [1]:
# 03_baseline_models.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 03 - Baseline predictive models.
#
# Trains four classifiers per dataset on the TRAIN split and reports discrimination
# on the TEST split: Logistic Regression, XGBoost, LightGBM, CatBoost. Class
# imbalance is handled with class weights / scale_pos_weight (no resampling).
#
# Reviewer-driven change: the single classifier carried into the RCFS experiments
# is named and fixed here. CANONICAL_MODEL = "XGBoost" with the exact XGB_PARAMS
# below; notebooks 04 and 05 reuse the identical configuration, split, and feature
# construction, so the baseline AUC in the RCFS tables equals the XGBoost row of
# the baseline-metrics table (Table 2) by construction. No paths are printed.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             balanced_accuracy_score, brier_score_loss)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
CANONICAL_MODEL = "XGBoost"
# Canonical XGBoost configuration, reused verbatim by notebooks 04 and 05.
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  random_state=RANDOM_STATE, n_jobs=-1)

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

DATASETS = {"German Credit": "german", "Taiwan CC Default": "taiwan",
            "Credit Card Approval": "credit_card_approval"}


def load_split(key, part):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_{part}.parquet"))


def get_Xy(df, key):
    meta = META[key]
    drop = [meta["target_col"]] + meta["sensitive_cols"]
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[meta["target_col"]].astype(int)
    return X, y


def build_models(y_train):
    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                                  random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(scale_pos_weight=spw, **XGB_PARAMS),
        "LightGBM": LGBMClassifier(n_estimators=400, max_depth=-1, learning_rate=0.05,
                                   subsample=0.9, colsample_bytree=0.9,
                                   class_weight="balanced", random_state=RANDOM_STATE,
                                   n_jobs=-1, verbose=-1),
        "CatBoost": CatBoostClassifier(iterations=400, depth=5, learning_rate=0.05,
                                       auto_class_weights="Balanced",
                                       random_seed=RANDOM_STATE, verbose=False),
    }


def evaluate(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"AUC": roc_auc_score(y_true, proba),
            "PR_AUC": average_precision_score(y_true, proba),
            "F1": f1_score(y_true, pred, zero_division=0),
            "BalancedAcc": balanced_accuracy_score(y_true, pred),
            "Brier": brier_score_loss(y_true, proba)}


results = []
predictions = {}
for display_name, key in DATASETS.items():
    train, test = load_split(key, "train"), load_split(key, "test")
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    predictions[key] = {}
    for model_name, model in build_models(y_train).items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        m = evaluate(y_test.values, proba)
        m.update({"dataset": display_name, "model": model_name})
        results.append(m)
        pred_df = test[META[key]["sensitive_cols"]].copy()
        pred_df["y_true"] = y_test.values
        pred_df["proba"] = proba
        predictions[key][model_name] = pred_df
        print(f"[{display_name:>22}] {model_name:<18} AUC={m['AUC']:.3f} "
              f"PR-AUC={m['PR_AUC']:.3f} F1={m['F1']:.3f}")

res_df = pd.DataFrame(results)[
    ["dataset", "model", "AUC", "PR_AUC", "F1", "BalancedAcc", "Brier"]].round(4)
print(res_df.to_string(index=False))
res_df.to_csv(os.path.join(TAB_DIR, "03_baseline_metrics.csv"), index=False)

# Persist test-set predictions of the canonical model for reference.
PRED_DIR = os.path.join(PROCESSED_DIR, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)
for key, model_preds in predictions.items():
    for model_name, pred_df in model_preds.items():
        pred_df.to_parquet(os.path.join(PRED_DIR, f"{key}__{model_name}.parquet"))
print(f"Canonical model for RCFS experiments: {CANONICAL_MODEL}")
print("Notebook 03 complete.")


[         German Credit] LogisticRegression AUC=0.791 PR-AUC=0.609 F1=0.614


[         German Credit] XGBoost            AUC=0.781 PR-AUC=0.610 F1=0.612


[         German Credit] LightGBM           AUC=0.774 PR-AUC=0.623 F1=0.553


[         German Credit] CatBoost           AUC=0.786 PR-AUC=0.648 F1=0.651
[     Taiwan CC Default] LogisticRegression AUC=0.710 PR-AUC=0.492 F1=0.463


[     Taiwan CC Default] XGBoost            AUC=0.770 PR-AUC=0.545 F1=0.532


...

[     Taiwan CC Default] CatBoost           AUC=0.776 PR-AUC=0.553 F1=0.540


[  Credit Card Approval] LogisticRegression AUC=0.546 PR-AUC=0.139 F1=0.208


[  Credit Card Approval] XGBoost            AUC=0.692 PR-AUC=0.283 F1=0.318


[  Credit Card Approval] LightGBM           AUC=0.715 PR-AUC=0.298 F1=0.339


[  Credit Card Approval] CatBoost           AUC=0.659 PR-AUC=0.241 F1=0.279
             dataset              model    AUC  PR_AUC     F1  BalancedAcc  Brier
       German Credit LogisticRegression 0.7907  0.6086 0.6144       0.7274 0.1967
       German Credit            XGBoost 0.7813  0.6095 0.6116       0.7226 0.1795
       German Credit           LightGBM 0.7742  0.6227 0.5528       0.6798 0.1986
       German Credit           CatBoost 0.7858  0.6484 0.6508       0.7524 0.1716
   Taiwan CC Default LogisticRegression 0.7101  0.4915 0.4630       0.6600 0.2079
   Taiwan CC Default            XGBoost 0.7696  0.5451 0.5318       0.7070 0.1721
   Taiwan CC Default           LightGBM 0.7698  0.5456 0.5250       0.7015 0.1677
   Taiwan CC Default           CatBoost 0.7758  0.5534 0.5399       0.7136 0.1781
Credit Card Approval LogisticRegression 0.5465  0.1395 0.2080       0.5300 0.2469
Credit Card Approval            XGBoost 0.6915  0.2832 0.3182       0.6418 0.1936
Credit Card Approval  